In [1]:
import os
import pickle
import pandas as pd
import numpy as np

In [2]:
data_dir = os.path.join('dataset','ciao_timestamp')

# `data_process.py`(가장 처음 데이터 생성 작업)
- ratings.pkl
    - [(user x item), rating] 정보를 담고 있는 csr matrix
- times.pkl
    - [(user x item), timestamp] 정보를 담고 있는 csr matrix
- category.pkl
    - [(item x category), bool] 정보를 담고 있는 csr matrix
    - col : category에 대한 one-hot encoding
- trust.pkl
    - [(user x user), trust] 정보를 담고 있는 csr matrix

# `dataProcess.py`(두번째 데이터 생성 작업)
1. `splitData`
    - `data_process.py`에서 생성한 데이터 load : [ratings, times, category, trust]
    - rating matrix에 대해서 row별로 순회하면서, interaction이 3미만인 경우 Drop
    - row별 interaction에 대해서 timestamp기준으로 내림차순 정렬(= 최근 먼저) 후에, non-zero인 경우만 추출
        - [rating matrix / time matrix] 모두
    - 그 이후, 가장 최근 interaction에 대해서 leave-last one-out기법으로 train/valid/test (= n-2:1:1) 분리 
    - csc matrix로 변환 후 저장
        - [train, valid, test, train_time, trust, category]

2. `filterData`
    - trust matrix에 대해서, symmetric으로 변형
        - train matrix에서 user별로 interaction이 존재하지 않는 경우, 해당 user Drop(Train, valid, test, train_time 모두)
        - train matrix에서 item별로 interaction이 존재하지 않는 경우, 해당 item Drop(Train, valid, test, train_time 모두)
        - trust matrix에서 user별로 interaction이 존재하지 않는 경우, 해당 user Drop(Train, valid, test, train_time 모두)
    - [train, valid, test, train_time, trust, category] 저장

3. `splitAgain`
    - [train, test, train_time] Load 후, lil matrix로 변환
    - test matrix에서 interaction이 없는 user에 대해서, train matrix에서 해당 user의 interaction이 2개 이상인 경우애, \
    train의 가장 마지막 interaction을 test matrix의 interaction으로 옮김
        - train[last_interaction] = 0
        - test[last_interaction] = 1
    - csr_matrix로 변환 후, 저장

4. `filterData`

5. `testNegSample`
    - [train, valid, test] load
    - [test, valid]에서 100개씩 negative sampling진행 

6. `createCategoryDict`
    - [train, category] load
    - {product_id : [category index]}의 형태로 dictionary 생성 후, 저장
        - categoryDict

7. `creatMultiItemUserAdj`(`create_adj.py`)
    - [train, train_time] load
    - multi_adj matrix 생성
        - row : rating별(1~5)로 구분한 product_id => (5 * # of products)
        - col : user id
        - data(value) : timestamp
    - multi_adj를 다른 zero-matrix와 합쳐 bipartite graph adjacency matrix 생성
        - nodes = user_id ∪ product_id
        - edges = "user가 특정 product를 rating한 행위" (timestamp로 가중치 표현).

8. `generateGraph`
    - [train, trust, category, categoryDict] Load
    - UU matrix
        - trust를 기반으로한 user-user adjacency matrix
    - II matrix
        - category matrix에서 item 별로 동일한 category를 가지는 item list를 추출 
        - 그 중에서 random sample -> item list2
        - anchor item과 연결 관계가 있는 item을 연결하는 adjacency matrix생성
            - ITI matrix
        - symmetric matrix로 변환
    - uu_vv_graph에 dictionary 형태로 저장


# rmse, 

# model - init
1. getData를 통해서, 위에서 생성한 data load
    - train matrix, valid data, multi_adj_item, uu matrix, vv matrix
2. uu matrix, vv matrix를 기반으로 DGI를 통해 Graph data 생성
3. 각 graph별로 sub-graph(?) 생성
4. multi_adj_item matrix에 대해서, timestamp를 normalize해 multi_adj_time_norm matrix생성
5. 생성된 multi_adj_time_norm를 통해서 uv_g(Graph data)생성
6. train data에 대해, negative sampling하면서 train set/loader 생성
    - valid/test는 이미 negative sampling되어 있으므로, 그냥 valid/test set/loader 생성

# model - run
- prepareModel()
    - 

In [ ]:
# 1. getData ; dataProcess.py에서 생성한 [train(matrix), test data(user-item), valid data(user-item), train_time(matrix), trust(matrix), multi_adj_matrix, uu_vv_graph] load
# 실제로 사용하는 것 : [train matrix, valid data, multi_adj_item, uu matrix, vv matrix]
with open(os.path.join(data_dir, 'uu_vv_graph.pkl'), 'rb') as f:
    uu_vv_Graph = pickle.load(f)
    
with open(os.path.join(data_dir, 'multi_item_adj.pkl'), 'rb') as f:
    multi_adj_item = pickle.load(f)